# 第3回：回帰・分類・前処理Pipelineでモデルを作る

この回は3つのパートで構成します：**数値を予測する—回帰 ／ クラスを予測する—分類 ／ 前処理をPipelineにまとめる**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼って
説明や修正を相談します（`ASK COPILOT`）。ただしAIの答えは鵜呑みにせず、必ず自分の出力で確かめます。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向け、
`DEEP DIVE`・`APPENDIX`は発展です（飛ばしても本編は完結します）。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

連続値の回帰、クラスの分類、そして数値列とカテゴリ列を安全に扱うPipelineで、評価できるモデルを組み立てます。

- MAE・RMSE・R²を異なる視点として読み、複数モデルを同条件で比較する
- 学習曲線と残差診断から、データ不足か表現力不足かを切り分ける
- 分位点回帰やブートストラップで予測の不確かさを区間として示す
- 混同行列とprecision・recall・F1・PR-AUCを利用場面へ結びつける
- 確率の較正（calibration）を信頼度図と指標で評価する
- 不均衡データへclass_weightや閾値調整で対処し、効果を検証する
- 列型ごとの前処理をColumnTransformerで分け、Pipelineへ一体化する
- BaseEstimatorとTransformerMixinで、意味のある自作変換器を書く
- 前処理の選択肢をGridSearchCVの探索対象に含める

### この回の進み方（大切）

この回は、旧カリキュラムの**3回分をまとめた長い回**です。**パート1→2→3**の順に、各パートの
`CORE`（本線）で手を動かします。1回の時間で全部を終える必要はありません。各パートの
`DEEP DIVE`／`APPENDIX`は、余裕のある人や自習で進めてください。日をまたいで少しずつでも大丈夫です。

### 先に押さえる言葉

- MAE：絶対誤差の平均
- RMSE：大きな誤差をより重く扱う指標
- 残差：実測値と予測値の差
- 学習曲線：データ量に対する性能の変化
- 予測区間：予測値に付ける不確かさの幅
- precision：陽性予測のうち正しかった割合
- recall：実際の陽性を見つけた割合
- PR-AUC：適合率-再現率曲線の下側面積
- 較正：予測確率と実際の頻度が一致している度合い
- class_weight：少数クラスの誤りを重く扱う設定
- ColumnTransformer：列ごとに別の前処理を割り当てる仕組み
- 自作変換器：fit/transformを実装した独自の前処理
- get_feature_names_out：変換後の列名を取得するAPI
- パラメータ探索：前処理やモデルの設定を系統的に比較すること
- メモリキャッシュ：共通の前処理計算を使い回す仕組み

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：数値を予測する—回帰

**このパートの問い：連続値の予測モデルを、何と比べ、不確かさをどう示すか。**


## 回帰＝「数値そのもの」を予測する

ここまでは活性の有無（0/1）でしたが、この回は**収率（%）という連続値**を予測します。これを
**回帰**と呼びます。回帰でいちばん大事な問いは「その予測は**何と比べて**良いのか」。だから今回も
**平均値だけを返すベースライン**を必ず土俵に上げます。

まず下準備。日本語フォント設定と、使うモデルの読み込み、学習/検証の分割をまとめて行います。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["yield_pct"], test_size=0.25, random_state=42)


## TRY：4モデルを交差検証で、3つの指標で比べる

回帰の代表的な指標を先に押さえます。

- **MAE（平均絶対誤差）**：平均で何%外すか。単位が収率と同じで**いちばん直感的**。
- **RMSE**：大きな外れを二乗で重く見る。「たまの大外し」を嫌う場面向け。
- **R²（決定係数）**：平均値予測と比べてどれだけ説明できたか（1に近いほど良い、0は平均値並み）。

`neg_...`はsklearnの都合で「大きいほど良い」に符号反転された指標名。表示時に`-`で元へ戻します。
（scikit-learnの`scoring`は「大きいほど良い」に統一されているため、誤差系の指標は符号が反転しています。）

コード中の`make_pipeline(SimpleImputer(...), モデル)`は、**欠損補完とモデルを1つにまとめて「1個のモデル」の
ように扱う**ための道具です。こうすると`fit`/`predict`や交差検証がまとめて安全に回せます。仕組みは第9回で
詳しく学ぶので、ここでは「前処理とモデルをセットにする書き方」とだけ捉えて大丈夫です。


In [ ]:
from sklearn.model_selection import cross_validate, KFold

models = {
    "平均値": DummyRegressor(),
    "線形回帰": make_pipeline(SimpleImputer(strategy="median"), LinearRegression()),
    "決定木": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
}
cv = KFold(5, shuffle=True, random_state=42)
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
rows = []
for name, estimator in models.items():
    res = cross_validate(estimator, df[features], df["yield_pct"], cv=cv, scoring=scoring)
    rows.append({"モデル": name, "MAE": -res["test_MAE"].mean(), "RMSE": -res["test_RMSE"].mean(), "R2": res["test_R2"].mean()})
pd.DataFrame(rows).sort_values("MAE").round(3)


### 出力の読み方

- MAEの小さい順に並びます。**「平均値」より各モデルがどれだけMAEを下げたか**が価値です。下げ幅が小さいなら、その特徴量では収率を説明しきれていません。
- MAEとRMSEの差が大きいモデルは、**たまに大きく外している**サイン（RMSEが大外れを強調するため）。
- R²が0近くなら「平均値と大差ない」、負なら「平均値より悪い」。**まずベースライン超え**を確認します。
- なお、この`cross_validate`は内部でデータを分割し直して評価します。上のセルで作った`X_train`/`X_valid`はここでは使わず、この後の**残差図**（予測と実測を見る図）で使います。


## 予測と実測、そして「残差」を絵で見る

数字だけでなく図で確かめます。左は**予測と実測の散布図**（点が対角線に乗るほど良い）、右は
**残差図**（実測−予測を予測値に対してプロット）。残差は0の周りに**模様なくばらける**のが理想です。
偏りや傾きがあれば、モデルが取りこぼした構造があります。


In [ ]:
rf = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)).fit(X_train, y_train)
pred = rf.predict(X_valid)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_valid, pred, alpha=0.6)
axes[0].plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], "--")
axes[0].set(xlabel="実測収率", ylabel="予測収率", title="予測と実測")
axes[1].scatter(pred, y_valid - pred, alpha=0.6)
axes[1].axhline(0, linestyle="--")
axes[1].set(xlabel="予測収率", ylabel="残差（実測-予測）", title="残差")
plt.tight_layout()


### 出力の読み方

- **左**：点が破線（予測＝実測）に近いほど良い。高収率の領域で点が下に外れていれば、**高い収率を低めに予測しがち**という癖です。
- **右**：残差が予測値によって偏る（例：予測が大きいほど残差が下がる）なら、モデルが端の領域を苦手にしています。**きれいな水平の帯**が理想。
- 図は「どこで外すか」を教えてくれます。次のセルで、実際に大きく外した試料を取り出します。


## 大きく外した試料を名指しで調べる

平均のMAEでは見えない「個別の大外し」を確認します。絶対誤差の大きい上位8件を、系列や触媒つきで
取り出すと、**特定の条件で外していないか**の手がかりになります。


In [ ]:
errors = df.loc[y_valid.index, ["sample_id", "scaffold_group", "catalyst", "yield_pct"]].copy()
errors["予測"] = pred
errors["絶対誤差"] = (errors["yield_pct"] - errors["予測"]).abs()
errors.nlargest(8, "絶対誤差").round(2)


### 出力の読み方

大外し8件に**同じ系列や同じ触媒が偏っていないか**を見ます。偏っていれば、その条件を表す特徴量が
足りない可能性（第11回の特徴量設計の動機）。ばらばらなら、単なるノイズかもしれません。

## CHANGE

`max_depth=6`を`3`や`10`へ変え、MAEの表・残差図・大外し試料が**どう連動して動くか**を観察します。


## DEEP DIVE：伸び悩みの原因と、予測の不確かさ

発展として3つ。**学習曲線**（データを増やせば改善するか）、**群別残差**（どの系列で系統的に外すか）、
**予測区間**（1点の予測に幅を添える）です。


### 学習曲線：データ不足か、表現力不足か

「もっとデータを集めれば精度が上がる？」に答える図です。学習データ量を増やしながら、学習MAEと
検証MAEの推移を描きます。2本が近づいて高止まりなら**データ追加は効きにくい**（特徴量やモデルを
見直すべき）。2本が離れて検証MAEがまだ下がりそうなら**データ追加が効く**サインです。


In [ ]:
import numpy as np
from sklearn.model_selection import learning_curve

sizes, train_scores, valid_scores = learning_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)),
    df[features], df["yield_pct"], cv=5, scoring="neg_mean_absolute_error",
    train_sizes=np.linspace(0.2, 1.0, 5),
)
plt.figure(figsize=(7, 4))
plt.plot(sizes, -train_scores.mean(1), "o-", label="学習MAE")
plt.plot(sizes, -valid_scores.mean(1), "o-", label="検証MAE")
plt.xlabel("学習データ件数"); plt.ylabel("MAE"); plt.legend(); plt.title("学習曲線")
plt.tight_layout()
print("2本が高止まりで近いなら、データ追加より特徴量やモデルを見直す。")


### 出力の読み方

- 右端（全データ使用）で**検証MAEがまだ下降中**なら、データを増やす価値あり。**水平に寝ている**なら頭打ち。
- 学習MAEと検証MAEの**縦の隙間**が過学習の程度。隙間が大きいほど「覚えすぎ」寄りです。


### 群別に残差を見る：どの系列で偏るか

全体のMAEが良くても、特定の化合物系列だけ系統的に外していることがあります。系列ごとに件数・MAE・
**平均残差**（符号つき）を出すと、「この系列を平均的に低く見積もっている」といった偏りが見えます。


In [ ]:
errors["残差"] = errors["yield_pct"] - errors["予測"]
group_error = errors.groupby("scaffold_group").agg(件数=("残差", "size"), MAE=("絶対誤差", "mean"), 平均残差=("残差", "mean"))
display(group_error.sort_values("MAE", ascending=False).round(2))
print("平均残差が正なら、その系列を平均的に過小予測している。")


### 出力の読み方

**平均残差が0から大きく離れた系列**が要注意。正なら過小予測、負なら過大予測です。件数が少ない系列は
偶然も大きいので、件数と併せて読みます。系統的な偏りは、その系列を表す特徴量の不足を示唆します。


### 予測区間：1点でなく「幅」で答える

「収率は62%」より「10〜90%の確率で50〜74%」の方が、意思決定に誠実なことがあります。**分位点回帰**で
下限（10%点）と上限（90%点）を別々に予測し、実測が本当にその区間に入る割合（**被覆率**）を検証します。


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

low = HistGradientBoostingRegressor(loss="quantile", quantile=0.1, max_iter=200, random_state=42).fit(X_train, y_train)
high = HistGradientBoostingRegressor(loss="quantile", quantile=0.9, max_iter=200, random_state=42).fit(X_train, y_train)
low_pred, high_pred = low.predict(X_valid), high.predict(X_valid)
coverage = ((y_valid.to_numpy() >= low_pred) & (y_valid.to_numpy() <= high_pred)).mean()
print(f"10-90%予測区間の実測被覆率: {coverage:.1%}（理想は約80%）")
pd.DataFrame({"実測": y_valid.to_numpy()[:8], "下限": low_pred[:8].round(1), "上限": high_pred[:8].round(1)})


### 出力の読み方

- 10〜90%区間なので、被覆率は**理想80%**に近いほど区間が正直。大きく下回れば区間が狭すぎ（自信過剰）、上回れば広すぎです。
- 表の8件で、**実測が下限〜上限に収まっているか**を目で確認します。幅の広い試料はモデルが自信を持てていない試料です。


## APPENDIX（任意・追加演習）

線形モデルの正則化や非線形化を試します。90分の外の自習向けです。まず**RidgeとLasso**を、正則化の
強さ`alpha`を変えて比較します（`alpha`が大きいほど係数を抑え、過学習を防ぐ）。


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(5, shuffle=True, random_state=42)
rows = []
for alpha in [0.01, 0.1, 1.0, 10.0]:
    for name, reg in {"Ridge": Ridge(alpha=alpha), "Lasso": Lasso(alpha=alpha, max_iter=5000)}.items():
        pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), reg)
        mae = -cross_val_score(pipe, df[features], df["yield_pct"], cv=cv, scoring="neg_mean_absolute_error").mean()
        rows.append({"モデル": name, "alpha": alpha, "MAE": mae})
pd.DataFrame(rows).pivot(index="alpha", columns="モデル", values="MAE").round(3)


### 出力の読み方

`alpha`を変えるとMAEが変わり、最適な強さがあることが分かります。強すぎると単純になりすぎ（未学習）、
弱すぎると過学習寄り。RidgeとLassoで最適`alpha`が違うのも普通です。**正則化は複雑さを調整するダイヤル**です。


### 多項式特徴量で「曲がり」を線形モデルに教える

線形回帰は直線しか引けませんが、`PolynomialFeatures`で二乗や交互作用の列を足すと、曲がった関係も
表せます。次数を上げすぎると過学習するので、交差検証で確かめます。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

for degree in [1, 2, 3]:
    pipe = make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        PolynomialFeatures(degree, include_bias=False), LinearRegression(),
    )
    mae = -cross_val_score(pipe, df[features], df["yield_pct"], cv=cv, scoring="neg_mean_absolute_error").mean()
    print(f"多項式次数{degree}: MAE={mae:.3f}")


### 出力の読み方

次数2で、温度の山型（第4回）を線形モデルが表せるようになり、MAEが下がることが多いはず。ただし次数3で
悪化したら過学習のサイン。「複雑にすれば良い」ではなく、**交差検証が下がる範囲でだけ複雑にする**が原則です。


### 部分依存プロット：モデルは各変数をどう使っているか

`PartialDependenceDisplay`は、「他を平均的に保ったまま、ある変数を動かすと予測がどう変わるか」を
描きます。モデルが温度の山型を学べているかを、目で確認できます。


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

rf = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)).fit(df[features], df["yield_pct"])
PartialDependenceDisplay.from_estimator(rf, df[features], ["temperature_c", "concentration_m"])
plt.tight_layout()


### 出力の読み方

温度の曲線が**山型**（中ほどで予測収率が最大）になっていれば、モデルは第4回で見た構造を学べています。
部分依存プロットは、ブラックボックスに見える木モデルの「考え方」を説明する強力な道具で、
研究者への説明資料としても有効です。


---

# パート2：クラスを予測する—分類

**このパートの問い：正解率だけで十分なのはどんなときか。**


## 「正解率」だけ見ると、なぜ危ないのか

第5回で「多数派と答えるだけで正解率が高くなる」ことを見ました。この回はその続きで、
**正解率（accuracy）に代わる読み方**を身につけます。鍵になるのが4つの結果です。

- **真陽性(TP)**：活性を活性と当てた／**真陰性(TN)**：非活性を非活性と当てた
- **偽陽性(FP)**：非活性を活性と誤った（無駄な追試）／**偽陰性(FN)**：活性を見逃した（機会損失）

この4つを表にしたのが**混同行列**です。まずロジスティック回帰を学習し、各試料の**活性確率**を出します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## TRY：閾値0.5で混同行列と3指標を読む

確率が0.5以上なら「活性」と判定し、結果を混同行列で見ます。同時に3つの指標を出します。

- **precision（適合率）**：活性と判定したうち、本当に活性だった割合（＝空振りの少なさ）。
- **recall（再現率）**：本当の活性のうち、見つけられた割合（＝見逃しの少なさ）。
- **F1**：precisionとrecallのバランス（両方が高いときだけ高くなる）。


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


### 出力の読み方

- 混同行列は**左上=TN、右下=TP**が当たり、**右上=FP、左下=FN**が外れ。色が濃い（数が多い）マスに注目します。
- **accuracyは高いのにrecallが低い**、という組み合わせが起きがちです。これは「非活性はよく当てるが、肝心の活性を見逃している」状態。活性が少ないデータでは、accuracyが良く見えてもこの罠にはまります。
- 「何を重視するか」で読む指標が変わる、というのが今回いちばん覚えておいてほしい点です。


## TRY：判定の「閾値」を動かしてみる

0.5は絶対ではありません。閾値を下げると「活性」と判定する数が増え、**recallは上がるがprecisionは下がる**、
というトレードオフが起きます。0.3・0.5・0.7で指標がどう動くか並べます。


In [ ]:
rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


### 出力の読み方

- 閾値を下げる（0.3）と**recallが上がりprecisionが下がる**、上げる（0.7）と逆。表で必ずこの向きになるはずです。
- **見逃しを避けたい場面は閾値を下げ、空振りを避けたい場面は上げる**。閾値はモデルの外側で、目的に合わせて選ぶダイヤルです（第5回のコスト最適閾値につながります）。


## CORE深掘り：不均衡データではPR-AUCを見る

「閾値をいくつにするか」を決める前に、モデルの**確率の質そのもの**を1つの数字で測りたい。不均衡データ
（活性が少ない）では、ROC-AUCよりも**PR-AUC（適合率-再現率曲線の面積）**の方が実態を映します。


In [ ]:
from sklearn.metrics import average_precision_score
print("活性の割合:", round(df["active"].mean(), 3))
print("PR-AUC(平均適合率):", round(average_precision_score(y_valid, probability), 3))
print("常に多数派と予測したときのaccuracy:", round((y_valid == y_valid.mode()[0]).mean(), 3))


### 出力の読み方

- 「活性の割合」が小さいのに「多数派予測のaccuracy」が高い——**accuracyの水増し**を数字で確認できます。
- **PR-AUC**は「活性の割合」を基準線とし、それを大きく上回るほど、モデルが活性をうまく上位に並べていると読めます。閾値を決めずにモデルの良さを比べたいときの主指標です。


## 話し合い

「見逃し（FN）と空振り（FP）の、どちらがこのテーマでは高くつくか？」を5人で言葉にします。
探索段階なら見逃しを嫌ってrecall寄り、確証段階なら空振りを嫌ってprecision寄り。**正解は場面で変わります。**


## DEEP DIVE：確率を「信じてよいか」と、不均衡対策

確率をコストの計算（第5回）に使うなら、その確率が**較正**されている——「0.8と言ったら本当に約80%」で
ある必要があります。ここでは較正の測り方と直し方、そして少数クラスへの対処を扱います。


### 較正：予測確率と実際の頻度は一致しているか

**信頼度図**は、予測確率（横軸）に対して実際の活性率（縦軸）を描き、対角線に近いほど較正が良い、と
読みます。**Brierスコア**は較正のズレを1つの数字にしたもの（小さいほど良い）。`CalibratedClassifierCV`で
較正し直し、前後を比べます。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

base_clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
cal_clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5).fit(X_train, y_train)
plt.figure(figsize=(6, 5))
for name, clf in {"未較正": base_clf, "較正後": cal_clf}.items():
    p = clf.predict_proba(X_valid)[:, 1]
    print(f"{name}: Brier={brier_score_loss(y_valid, p):.3f}（小さいほど良い）")
    frac, mean_pred = calibration_curve(y_valid, p, n_bins=5)
    plt.plot(mean_pred, frac, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("予測確率"); plt.ylabel("実際の頻度"); plt.legend(); plt.title("信頼度図")
plt.tight_layout()


### 出力の読み方

- 折れ線が**対角線（点線）に近い**ほど較正が良好。対角線から膨らんでいれば、その確率帯で自信過剰／過小です。
- Brierが較正後に下がっていれば改善成功。ただし小さいデータでは較正が不安定なこともあるので、図と数字の両方で判断します。
- なお`base_clf`は「未較正」の比較用に学習しています。`CalibratedClassifierCV`は`cv=5`を指定しているため内部でモデルを学習し直します（`base_clf`の学習結果そのものは較正には使いません）。


### コスト行列で閾値を決める（較正済み確率で）

第5回と同じ考え方を、較正した確率に適用します。見逃し(FN)が空振り(FP)の8倍高いとして、期待コストが
最小の閾値を探します。**較正済みの確率**を使うことで、コスト計算の前提が整います。


In [ ]:
import numpy as np
proba_cal = cal_clf.predict_proba(X_valid)[:, 1]
cost_fn, cost_fp = 8, 1
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba_cal >= t).astype(int)
    fp = int(((pred == 1) & (y_valid == 0)).sum())
    fn = int(((pred == 0) & (y_valid == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
table = pd.DataFrame(rows)
print("コスト最小の閾値:", table.loc[table["期待コスト"].idxmin(), "閾値"])
table


### 出力の読み方

見逃しのコストが高いので、最適閾値は**0.5より低め**に出ます。「0.5で判定」がいかに恣意的かを、
ここでも数字で確認できます。コスト比を変えれば最適点も動きます。


### class_weight：少数クラスの誤りを重く扱う

閾値調整とは別に、**学習の時点で**少数クラス（活性）の誤りを重く扱う方法があります。`class_weight="balanced"`は
少数クラスを自動で重み付けします。recall（見逃しの少なさ）がどう変わるかを見ます。


In [ ]:
for label, weight in {"weightなし": None, "balanced": "balanced"}.items():
    clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000, class_weight=weight)).fit(X_train, y_train)
    pred = clf.predict(X_valid)
    print(f"{label:10s} F1={f1_score(y_valid, pred):.3f}  recall={recall_score(y_valid, pred):.3f}")


### 出力の読み方

`balanced`にすると**recallが上がりやすい**（活性を見つけにいく）反面、precisionやF1は下がることもあります。
「閾値で調整」と「重みで調整」は似た効果を持つ別の道具。どちらが目的に合うかを、指標を見て選びます。


## APPENDIX（任意・追加演習）

分類の評価を、曲線と閾値でさらに掘り下げます。90分の外の自習向けです。まず2モデルの
**適合率-再現率曲線**と**ROC曲線**を並べます。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

candidates = {
    "ロジスティック": make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, est in candidates.items():
    est.fit(X_train, y_train)
    PrecisionRecallDisplay.from_estimator(est, X_valid, y_valid, ax=axes[0], name=name)
    RocCurveDisplay.from_estimator(est, X_valid, y_valid, ax=axes[1], name=name)
axes[0].set_title("適合率-再現率曲線"); axes[1].set_title("ROC曲線")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
plt.tight_layout()


### 出力の読み方

- **PR曲線**は右上に張り付くほど良い。不均衡データでは、この曲線とその面積(PR-AUC)がROCより実態を映します。
- **ROC曲線**は左上に張り付くほど良い。凡例のAUCで一目比較できます。
- 2モデルの曲線が交差するなら、**どの動作点（閾値）で使うかによって優劣が変わる**ということです。


### 目標recallを満たす閾値を逆算する

「活性の見逃しは9割以上防ぎたい（recall≧0.9）」のような要件から、それを満たしつつprecisionが最大の
閾値を選びます。要件を先に決め、閾値を後から合わせる実務的なやり方です。


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

proba = candidates["Random Forest"].predict_proba(X_valid)[:, 1]
prec, rec, thr = precision_recall_curve(y_valid, proba)
target_recall = 0.9
ok = rec[:-1] >= target_recall
if ok.any():
    idx = np.argmax(np.where(ok, prec[:-1], -1))
    print(f"recall>={target_recall} を満たす閾値: {thr[idx]:.3f}  precision={prec[idx]:.3f}  recall={rec[idx]:.3f}")
else:
    print("目標recallを満たす点がありません")


### 出力の読み方

選ばれた閾値は0.5より低いはず（見逃しを減らすには「活性」と判定する範囲を広げる）。その代償に
precisionが下がります。**要件→閾値**の順で決めると、恣意的な0.5から卒業できます。


### 交差検証で混同行列を集計する

1回の検証ではなく、`cross_val_predict`で全データのOOF予測を作り、混同行列を集計します。1回分より
安定した内訳が見えます。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix

oof = cross_val_predict(
    make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    df[features], df["active"], cv=StratifiedKFold(5, shuffle=True, random_state=42),
)
cm = confusion_matrix(df["active"], oof)
display(pd.DataFrame(cm, index=["実:非活性", "実:活性"], columns=["予:非活性", "予:活性"]))


### 出力の読み方

全420件を1件ずつ「その行を学習に使わないモデル」で予測した集計です。右上（偽陽性）と左下（偽陰性）の
大きさを比べ、**このモデルがどちらの誤りをしやすいか**を把握します。改善の方向づけに使えます。


---

# パート3：前処理をPipelineにまとめる

**このパートの問い：数値列とカテゴリ列を、安全に同じモデルへ入れるにはどうするか。**


## なぜ「Pipeline」が必要なのか

これまで欠損を`fillna`で埋めたり、数値だけを使ったりしてきました。実データでは**数値列と
カテゴリ列（文字）が混在**し、それぞれ別の下ごしらえが要ります。

- 数値列 → 欠損を埋める＋尺度を揃える（標準化）
- カテゴリ列 → 欠損を埋める＋数値へ変換（One-Hot：各カテゴリを0/1の列にする）

これらを手作業でやると、**第6回で学んだ前処理リーク**（検証情報の漏れ）を起こしがちです。そこで
`Pipeline`と`ColumnTransformer`を使い、**前処理からモデルまでを1つの部品**にまとめます。こうすると
交差検証や予測のたびに、前処理が正しく分割の内側で学習されます。

まず数値列・カテゴリ列を決め、学習/検証に分けます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

numeric = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
categorical = ["solvent", "catalyst", "scaffold_group"]
X = df[numeric + categorical]
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## TRY：列ごとの前処理を組み立てて、モデルまで繋ぐ

各部品の役割：

- `numeric_process`：数値列の下ごしらえ（欠損補完→標準化）を並べた小さなPipeline。
- `categorical_process`：カテゴリ列の下ごしらえ（欠損補完→One-Hot）。
- `ColumnTransformer`：「この列たちには数値処理、あの列たちにはカテゴリ処理」と**列ごとに担当を割り当てる**部品。
- 最後に`Pipeline([("前処理", preprocess), ("予測", ロジスティック回帰)])`で**前処理＋モデルを一体化**。

`model.fit`一発で、前処理もモデルもまとめて学習されます。


In [ ]:
numeric_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="median")),
    ("標準化", StandardScaler()),
])
categorical_process = Pipeline([
    ("欠損補完", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("数値列", numeric_process, numeric),
    ("カテゴリ列", categorical_process, categorical),
])
model = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
print(classification_report(y_valid, model.predict(X_valid), target_names=["非活性", "活性"]))


### 出力の読み方

`classification_report`は、クラスごとにprecision・recall・F1と件数(support)を並べた総合成績表です。

- **活性クラスの行**を重点的に見ます（少数派で難しいため）。第8回で学んだとおり、accuracyより各クラスのrecall/precisionが実態を映します。
- 大事なのは点数そのものより、**文字列カテゴリを含む表をエラーなく1つのモデルへ通せた**こと。手作業のOne-Hotより安全で短いです。

補足：ここでは`scaffold_group`（化合物系列）もカテゴリ列の例として入れていますが、第6回のとおり本来は
**系列を跨がない分割（GroupKFold）とセットで扱うべき列**です。この回はPipelineの組み方の説明が目的なので
乱数分割のまま使っていますが、実データで系列をカテゴリ特徴量にするときは、この点に注意してください。


## 未知カテゴリが来ても止まらない

本番では、学習時に無かった溶媒名が来ることがあります。`OneHotEncoder(handle_unknown="ignore")`の
おかげで、未知カテゴリでもエラーにならず予測できます。わざと存在しない溶媒名を入れて確かめます。


In [ ]:
unknown = X_valid.iloc[[0]].copy()
unknown["solvent"] = "New-Solvent"
print("未知カテゴリを含む予測:", model.predict(unknown)[0])


### 出力の読み方

エラーで止まらず予測が返れば成功です。`handle_unknown="ignore"`が無いと、未知カテゴリで例外が出て
本番が止まります。ここで身につけてほしいのは、**「本番で起きうる入力」を想定して前処理を設計する**という実務感覚です。


## CORE深掘り：変換後は列が増える。その姿を見る

One-Hotはカテゴリごとに0/1の列を作るので、**列数が増えます**。`get_feature_names_out`で変換後の列名を、
`transform`で実際の数値を確認し、Pipelineの中で何が起きているかを可視化します。


In [ ]:
names = model.named_steps["前処理"].get_feature_names_out()
transformed = model.named_steps["前処理"].transform(X_train.head(3))
if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()
print("元の列数:", X_train.shape[1], "→ 変換後:", transformed.shape[1])
pd.DataFrame(transformed, columns=names, index=X_train.head(3).index).iloc[:, :10].round(2)


### 出力の読み方

- **元の列数 → 変換後**で列が増えているのは、カテゴリがOne-Hotで展開されたため。列名に`カテゴリ列__solvent_EtOH`のような名前が付きます。
- 数値列は標準化され、**平均0付近・小さめの値**になっています。One-Hot列は0か1。「モデルが実際に見ている数字」はこの姿です。

## CHANGE

数値の欠損補完を`median`から`mean`へ変え、成績を比べます。変更は`SimpleImputer(strategy=...)`の**1か所だけ**。Pipelineだと変更点が1か所に集約され、実験が管理しやすくなります。


## DEEP DIVE：自作の前処理を作り、前処理も探索対象にする

sklearnに用意された変換だけでなく、**自分の化学知識を前処理として書く**ことができます。また、
「どの補完戦略が良いか」のような前処理の選択も、モデルの設定と同じく**交差検証で選べます**。


### 自作変換器：`fit`と`transform`を持つ部品を書く

`BaseEstimator, TransformerMixin`を継承し、`fit`（学習することがあれば覚える）と`transform`（変換する）を
実装すれば、**Pipelineに差し込める自分だけの前処理**になります。ここでは「分子量あたりのTPSA」と
「最適温度からの距離」を足す変換器を作ります。


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class ChemRatioFeatures(BaseEstimator, TransformerMixin):
    "分子量あたりのTPSAと、最適温度78℃からの距離を足す自作変換器。"
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["tpsa_per_mw"] = X["tpsa"] / X["molecular_weight"].replace(0, np.nan)
        X["temp_distance"] = (X["temperature_c"] - 78).abs()
        return X

ChemRatioFeatures().fit_transform(df[["tpsa", "molecular_weight", "temperature_c"]].head()).round(3)


### 出力の読み方

元の3列に、新しい2列（`tpsa_per_mw`・`temp_distance`）が加わっています。`fit`は何も学習せず自身を返す
だけ（この変換は統計量を使わないため）。この形にしておくと、`Pipeline`へ入れて**分割の内側で**適用でき、
第11回の特徴量設計をリークなく行えます。


### 前処理の設定を`GridSearchCV`で選ぶ

`Pipeline`の各部品の設定には`前処理__数値列__欠損補完__strategy`のように**アンダースコア2つ**で
辿り着けます。この記法を使い、補完戦略（median/mean）を交差検証で比較して自動選択します。


In [ ]:
from sklearn.model_selection import GridSearchCV

grid_pipe = Pipeline([("前処理", preprocess), ("予測", LogisticRegression(max_iter=1000))])
param_grid = {"前処理__数値列__欠損補完__strategy": ["median", "mean"]}
search = GridSearchCV(grid_pipe, param_grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

`best_params_`が選ばれた補完戦略、`best_score_`がそのときの交差検証F1です。ポイントは、**前処理も
モデル設定と同じ土俵で、リークなく比較・選択できる**こと。Pipelineにまとめておいたからこそ可能になります。


## APPENDIX（任意・追加演習）

Pipelineをさらに実務的に使い込みます。90分の外の自習向けです。まず`make_column_selector`で、
**列の型（数値/文字）から自動で担当を振り分ける**書き方。列名を手で並べる手間が消えます。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer

auto_pre = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include="object")),
)
auto_model = make_pipeline(auto_pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print("列の型で自動振り分けした検証精度:", round(auto_model.score(X_valid, y_valid), 3))


### 出力の読み方

`make_column_selector(dtype_include="number")`が数値列を、`"object"`が文字列列を自動で拾います。列が
増減しても書き換え不要。実データで列数が多いときに効きます。`.score`は分類では既定でaccuracyを返します。


### 前処理とモデルを「まとめて」探索する

前処理の設定とモデルのハイパーパラメータを、1つの`GridSearchCV`で同時に探します。すべてPipelineの
内側なので、リークなく公平に比較できます。


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

full = Pipeline([("前処理", preprocess), ("予測", RandomForestClassifier(random_state=42))])
grid = {
    "前処理__数値列__欠損補完__strategy": ["median", "mean"],
    "予測__max_depth": [4, 6, None],
    "予測__n_estimators": [200, 300],
}
search = GridSearchCV(full, grid, cv=5, scoring="f1")
search.fit(X_train, y_train)
print("最良設定:", search.best_params_)
print("最良CV F1:", round(search.best_score_, 3))


### 出力の読み方

前処理（補完戦略）とモデル（深さ・木の本数）の**最良の組み合わせ**が一度に選ばれます。組合せは
2×3×2=12通り×5分割=60回の学習。前処理も探索対象にできるのが、Pipeline最大の利点です。


### 学習済みPipelineを保存して再利用する

選ばれた最良のPipelineを`joblib`で保存し、読み直しても同じ予測になることを確かめます。前処理ごと
保存されるので、配布先は`predict`するだけです（第15回の永続化の先取り）。


In [ ]:
import joblib
import numpy as np

path = ROOT / "workspace" / "pipeline_09.joblib"
joblib.dump(search.best_estimator_, path)
loaded = joblib.load(path)
assert np.array_equal(search.best_estimator_.predict(X_valid), loaded.predict(X_valid)), "保存前後で予測が不一致"
print("保存・読込で同じ予測:", path)


### 出力の読み方

`assert`が通れば、前処理込みのPipelineが丸ごと保存・復元できたということ。「モデルだけ保存して前処理を
忘れる」という実務で頻発する事故を、Pipeline化で防げます。


---

## よくある誤り

- R²だけで利用可能と判断する
- テストデータでモデルを選ぶ
- 点予測だけを示し不確かさを伝えない
- 常に閾値0.5を使う
- 偽陽性と偽陰性のコストを同じとみなす
- 未較正の確率をそのまま意思決定へ使う
- 全データ平均で欠損補完する
- カテゴリを意味のない大小関係へ変換する
- 本番の未知カテゴリでエラーになる

## SELF-STUDY（任意・30〜60分）

- 学習曲線を描き、データ追加が効くかを1文で判断する
- 分位点回帰の10-90%区間の被覆率を検証データで確認する
- CalibratedClassifierCVで較正前後の信頼度図とBrierスコアを比べる
- コスト行列から期待コスト最小の閾値を求め、0.5と比較する
- 分子量あたりのTPSAを作る自作変換器を書き、Pipelineへ組み込む
- 数値標準化の有無と補完戦略をGridSearchCVで比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. MAEとRMSEは何を違って重視するか
2. 学習曲線から何を読み取れるか
3. 予測区間が点予測より役立つ場面はどこか
4. accuracyが危険な例は何か
5. 較正が悪い確率を使うと何が起きるか
6. コストから閾値をどう決めるか
7. 自作変換器に最低限必要なメソッドは何か
8. 前処理をPipelineへ入れるとリークがなぜ防げるか
9. get_feature_names_outは何に使うか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
